# Lesson 7, Part 3

Let's check how well our Taylor approximations for sine and cosine 
capture some basic geometric identities. 
The following code has been adapted from Part 1 of this Level 
to permit array computations. 
To ensure good convergence the needed Taylor order 
approaches 40, or 20 non-zero terms.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def taylor_cosine(theta,order=40):
    theta_square = theta * theta
    cosine = next_term = 1
    for k in np.arange(2,order+1,2):
        next_term *= theta_square / (k * (1-k))  #-(k-1) = 1-k
        cosine +=  next_term
    return cosine

def taylor_sine(theta,order=40):
    theta_square = theta * theta
    if np.isscalar(theta):
        sine = next_term = theta
    else:
        next_term = theta.copy()  # make new arrays
        sine = theta.copy()
    for k in np.arange(3,order+1,2):
        next_term *= theta_square / (k * (1-k))
        sine += next_term
    return sine
    
theta = np.linspace(0,np.pi/2,41)
diverge = np.abs(taylor_cosine(theta) - taylor_cosine(2*np.pi - theta))
print('Maximum divergence of estimated cos(2π-θ) from cos(θ) is','{:.0e}'.format(np.max(diverge)))
diverge = np.abs(taylor_sine(theta) + taylor_sine(2*np.pi - theta))
print('Maximum divergence of estimated sin(2π-θ) from -sin(θ) is','{:.0e}'.format(np.max(diverge)))
diverge = np.abs(taylor_cosine(theta) + taylor_cosine(np.pi - theta))
print('Maximum divergence of estimated cos(π-θ) from -cos(θ) is','{:.0e}'.format(np.max(diverge)))
diverge = np.abs(taylor_sine(theta) - taylor_sine(np.pi - theta))
print('Maximum divergence of estimated sin(π-θ) from sin(θ) is','{:.0e}'.format(np.max(diverge)))
diverge = np.abs(taylor_cosine(theta) - taylor_sine(np.pi/2 - theta))
print('Maximum divergence of estimated sin(π/2-θ) from cos(θ) is','{:.0e}'.format(np.max(diverge)))

As we've seen, Maclaurin series in *x* converge more quickly when *x* is small. 
This code  reduces any $\theta$ to a value between 0 and $\pi$/4 $\approx$s 0.79, 
in which case 9 Taylor terms suffice for convergence to near-float accuracy.

In [ ]:
def reduced_cosine(theta):
    theta = theta % (2 * np.pi)    # narrows θ to range 0 to 2π
    if theta > np.pi:
        theta = 2 * np.pi - theta  # narrows θ to range 0 to π
    switch_sign = 1
    if theta > np.pi/2:
        switch_sign = -1           # narrows θ to range 0 to π/2 with possible sign switch
        theta = np.pi - theta
    if theta > np.pi/4:            # narrows θ to range 0 to π/4
        return switch_sign * taylor_sine(np.pi/2 - theta,order=17)
    return switch_sign * taylor_cosine(theta,order=16)

def reduced_sine(theta):                 
    theta = theta % (2 * np.pi)    # narrows θ to range 0 to 2π
    switch_sign = 1   
    if theta > np.pi:              # narrows θ to range 0 to π with possible sign switch
        switch_sign = -1
        theta = 2 * np.pi - theta
    if theta > np.pi/2:            # narrows θ to range 0 to π
        theta = np.pi - theta 
    if theta > np.pi/4:            # narrows θ to range 0 to π/4
        return switch_sign * taylor_cosine(np.pi/2 - theta,order=16)
    return switch_sign * taylor_sine(theta,order=17)

max_diverge_cos = max_diverge_sin = 0
for theta in np.linspace(0,2*np.pi,41):
    diverge = abs(taylor_cosine(theta)-reduced_cosine(theta))
    if diverge > max_diverge_cos:
        max_diverge_cos = diverge
    diverge = abs(taylor_sine(theta)-reduced_sine(theta))
    if diverge > max_diverge_sin:
        max_diverge_sin = diverge
print("Maximum divergence between taylor_cosine and reduced_cosine is",'{:.0e}'.format(max_diverge_cos))
print("Maximum divergence between taylor_sine and reduced_sine is",'{:.0e}'.format(max_diverge_sin))

The previous code will not process ndarrays without explicit looping. 
The variants here will. 
They are harder to write than standard loops but run much faster with larger arrays.

In [ ]:
def reduced_cosine_alt(theta):
    if np.isscalar(theta):   # true if single number not formatted as list or array
        theta = np.array([theta])
        return (float(reduced_cosine_alt(theta)))
    theta = theta % (2 * np.pi)
    ix = theta > np.pi
    theta[ix] = 2 * np.pi - theta[ix]
    switch_sign = np.ones(len(theta))
    ix = theta > np.pi / 2
    switch_sign[ix] = -1
    theta[ix] = np.pi - theta[ix]
    ix = theta > np.pi / 4
    cosine = np.zeros(len(theta))
    cosine[ix] = switch_sign[ix] * taylor_sine(np.pi/2 - theta[ix],order=17)
    ix = np.logical_not(ix)          # indices with theta <= np.pi/4
    cosine[ix] = switch_sign[ix] * taylor_cosine(theta[ix],order=16)
    return cosine

def reduced_sine_alt(theta):
    if np.isscalar(theta):
        theta = np.array([theta])
        return (float(reduced_cosine_alt(theta)))
    theta = theta % (2 * np.pi)
    switch_sign = np.ones(len(theta))
    ix = theta > np.pi
    switch_sign[ix] = -1
    theta[ix] = 2 * np.pi - theta[ix]
    ix = theta > np.pi / 2
    theta[ix] = np.pi - theta[ix]
    sine = np.zeros(len(theta))
    ix = theta > np.pi/4
    sine[ix] = switch_sign[ix] * taylor_cosine(np.pi/2 - theta[ix],order=16)
    ix = np.logical_not(ix)
    sine[ix] = switch_sign[ix] * taylor_sine(theta[ix],order=17)
    return sine

theta = np.linspace(0,2*np.pi,41)
diverge = np.abs(taylor_cosine(theta) - reduced_cosine_alt(theta))
print("Maximum divergence between taylor_cosine and reduced_cosine_alt is",'{:.0e}'.format(np.max(diverge)))
diverge = np.abs(taylor_sine(theta) - reduced_sine_alt(theta))
print("Maximum divergence between taylor_sine and reduced_sine_alt is",'{:.0e}'.format(np.max(diverge)))

Let's check the relation cos 2$\theta$ = 2 cos<sup>2</sup> -1, which is a special case of deMoivre's Theorem. 

In [ ]:
theta = np.linspace(0,np.pi,41)
x = reduced_cosine_alt(2 * theta)
y = 2 * reduced_cosine_alt(theta)**2 - 1
rtol,atol = 0,1E-15
# np.allclose compares two arrays for similar values
close_enough = np.allclose(x,y,rtol,atol)  # True if and only if all |x-y| < atol + rtol * |y| 
print('Is estimated cos 2θ = 2 (cos θ)**2 - 1 within absolute tolerance',atol,\
      'and relative tolerance',rtol,'?',close_enough)

In principle, this lets us estimate cos $\theta$ from a shorter Taylor series for cos($\theta$/2<sup>*k*</sup>). 
In practice most advantages are outweighed by rounding errors. 
The math and numpy commands for **cos** and **sin** are far more efficient.

In [ ]:
def reduced_cosine_more(theta,power_of_2=4,order=8):
    if np.isscalar(theta):
        theta = np.array([theta])
        return (float(reduced_cosine_more(theta)))
    theta = theta % (2 * np.pi)
    theta[theta > np.pi] = 2 * np.pi - theta[theta > np.pi]
    switch_sign = np.sign(np.pi/2 - theta)
    theta[switch_sign<0] = np.pi - theta[switch_sign<0]
    theta /= 2**power_of_2
    cosine = taylor_cosine(theta,order)
    for i in range(power_of_2):
        cosine = 2 * cosine**2 - 1
    return switch_sign * cosine

print('Does dividing by 2**power allow a Taylor series of given order to come within tol of convergence?')
theta = np.linspace(0,2 * np.pi,41)
for power,order,atol in [(4,8,5E-14),(6,4,1E-9),(7,2,1E-5)]:
    close_enough = np.allclose(reduced_cosine_more(theta,power,order),taylor_cosine(theta),0,atol)
    print('power',power,', order',order,' tol',atol,':',close_enough)

The equation for cos 2$\theta$ implies cos($\theta$/2) = sqrt((1 + cos $\theta$) / 2) and sin($\theta$/2) = sqrt((1 - cos $\theta$) / 2). 
Through repeated applications in conjunction with deMoivre formulas for sums and differences, we can generate 
precise formulas for cosines and sines of any rational fraction of $\pi$. Here are some estimates constructed in this fashion.

In [ ]:
power_of_2 = 6
cosine = np.zeros(2*power_of_2+2)
sine = np.zeros(2*power_of_2+2)
cosine[0] = -1
print('  θ           cos θ                sin θ')
print('π /','{:2}'.format(1),':','{:15.14f}'.format(cosine[0]),'{:20.15f}'.format(sine[0]))
denom = 1
for k in range(1,power_of_2+1):
    denom *= 2
    cosine[k] = np.sqrt((1 + cosine[k-1])/2)
    sine[k] = np.sqrt((1 - cosine[k-1])/2)
    print('π /','{:2}'.format(denom),':','{:16.15f}'.format(cosine[k]),'{:20.15f}'.format(sine[k]))
cosine[power_of_2+1] = -1/2         # cos(2π/3)
sine[power_of_2+1] = np.sqrt(3/4)   # sin(2π/3)
denom = 1.5
for k in range(power_of_2+2,2*power_of_2+2):
    denom = int(denom * 2)
    cosine[k] = np.sqrt((1 + cosine[k-1])/2)
    sine[k] = np.sqrt((1 - cosine[k-1])/2)
    print('π /','{:2}'.format(denom),':','{:16.15f}'.format(cosine[k]),'{:20.15f}'.format(sine[k]))
    
cos_1div4,sin_1div4 = cosine[2],sine[2]
cos_1div8,sin_1div8 = cosine[3],sine[3]
cos_3div8 = cos_1div4 * cos_1div8 - sin_1div4 * sin_1div8
diverge = cos_3div8 - taylor_cosine(np.pi * 3/8)
print('\nDifference between cos_3div8 and taylor_cos(3π/8) is','{:.0e}'.format(diverge))
cos_1div6,sin_1div6 = cosine[-5],sine[-5]
cos_5div12 = cos_1div4 * cos_1div6 - sin_1div4 * sin_1div6
diverge = cos_5div12 - taylor_cosine(np.pi * 5/12)
print('Difference between cos_5div12 and taylor_cos(5π/12) is','{:.0e}'.format(diverge))

Next let's apply Snell's Law of refraction, which requires the computation of inverse sines. 
The code here uses a Newton-Raphson method to compute inverse sines 
and is accurate to near-float precision.

In [ ]:
def inv_sin(x,tol=1E-15):
    theta = np.pi/4
    diverge = 1/np.sqrt(2) - x
    while abs(diverge) > tol:
        theta -= diverge / reduced_cosine(theta)
        diverge = reduced_sine(theta) - x
    return theta

max_diverge = sum_diverge = 0
num = 50
for x in np.linspace(0,1,num):
    theta = inv_sin(x)
    diverge = abs(np.sin(theta)-x)
    sum_diverge += diverge
    if diverge > max_diverge:
        max_diverge = diverge
avg_diverge = sum_diverge / num

print('Absolute divergences between np.sin(inv_sin(x)) and x:')
print('Maximum','{:.0e}'.format(max_diverge),'   Mean','{:.0e}'.format(avg_diverge))

Let's compute angles of refraction across various interfaces and express them in degrees. 
While numpy has functions **degrees** to convert radians to degrees and **radians** to convert degrees to radians, 
let's write out the conversion factors explicitly.

In [ ]:
def angle_refract(angle_incidence,n_incidence,n_refract,isdegree=True):
    convert = np.pi/180 if isdegree else 1
    sine_incidence = reduced_sine(angle_incidence * convert)
    sine_refract = sine_incidence * n_incidence / n_refract
    return inv_sin(sine_refract) / convert 

# create dictionary of refraction indices
refract = {"air":1.0003,"water":1.333,"glass":1.55,"diamond":2.417} 

stuff1,stuff2 = 'air','water'
print('Angles in passage from',stuff1,'to',stuff2)
print("Incidence  Refraction  (in degrees)")
for angle1 in np.linspace(15,75,5):
    angle2 = angle_refract(angle1,refract[stuff1],refract[stuff2])
    print('{:5.0f}'.format(angle1),'{:12.2f}'.format(angle2))

Let's calculate critical angles for various interfaces. 
Beyond the critical angles, light will be reflected rather than refracted.

In [ ]:
def critical_angle(n_incidence,n_refract,isdegree=True):
    convert = np.pi/180 if isdegree else 1
    sine_refract = n_incidence / n_refract
    return inv_sin(sine_refract) / convert

refract = {"air":1.0003,"water":1.333,"glass":1.55,"diamond":2.417} 
    
print("Critical Angles in Degrees")
for stuff1 in ("air","water"):
    for stuff2 in ("water","glass","diamond"):
        if stuff1 != stuff2:
            critical = critical_angle(refract[stuff1],refract[stuff2])
            print(stuff2,'to',stuff1,':',round(critical,2))

Let's simulate the motion of a pendulum 
and plot the oscillations in starting angle (measured from vertical) over time.

In [ ]:
def simulate_pendulum(length,theta_0,gravity=9.807,circuits=10,num=1000):
    grav_to_length = gravity / length
    period_approx = 2 * np.pi / np.sqrt(grav_to_length)
    time = np.linspace(0,circuits*period_approx,num)
    dt = time[-1] / (num - 1)
    theta = np.zeros(num)   # angle at time t
    theta[0] = curr_theta = theta_0
    curr_velocity = 0
    for i in range(1,num):
        accel = - grav_to_length * taylor_sine(curr_theta)
        dv = accel * dt
        avg_velocity = curr_velocity + 0.5 * dv
        theta[i] = curr_theta = curr_theta + avg_velocity * dt
        curr_velocity += dv
    return time,theta

length,theta_0 = 1,np.pi/20
time,theta = simulate_pendulum(length,theta_0)
plt.title('Test Pendulum (First Pass)')
plt.plot(time,theta)
plt.show()

The oscillations are widening, violating the conservation of energy. 
Let's redo this with shorter intervals and better approximation within the interval.

In [ ]:
def simulate_pendulum_better(length,theta_0,gravity=9.807,circuits=10,num=1000):
    grav_to_length = gravity / length
    period_approx = 2 * np.pi / np.sqrt(grav_to_length)
    time = np.linspace(0,circuits*period_approx,num)
    dt = time[-1] / (num - 1)
    theta = np.zeros(num)   # angle at time t
    theta[0] = curr_theta = theta_0
    curr_velocity = 0
    for i in range(1,num):
        accel = - grav_to_length * taylor_sine(curr_theta)
        approx_dv = accel * dt
        approx_avg_vel = curr_velocity + 0.5 * approx_dv
        avg_theta = curr_theta + 0.5 * approx_avg_vel * dt
        avg_accel = - grav_to_length * taylor_sine(avg_theta)
        dv = avg_accel * dt
        avg_velocity = curr_velocity + 0.5 * dv
        theta[i] = curr_theta = curr_theta + avg_velocity * dt
        curr_velocity += dv
    return time,theta

length,theta_0 = 1,np.pi/20
time,theta = simulate_pendulum_better(length,theta_0)
expansion = np.max(theta[-len(theta)//10:-1])/theta_0
string1 = 'larger' if expansion > 1 else 'smaller'
print('Last peak is a fraction','{:.0e}'.format(abs(expansion-1)),string1,'than the first peak')
plt.title('Test Pendulum (Better)')
plt.plot(time,theta)
plt.show()

Let's measure the period of the pendulum. 
For a pendulum one meter long at the surface of the earth, 
a half-period is close to one second. 
The standard meter length was originally chosen to generate that result.

In [ ]:
def period_pendulum(time, theta):
    increasing = theta[1:] > theta[:-1]
    sign_change = increasing[1:] != increasing[:-1]
    # np.where(condition) selects indices that satisfy the condition 
    turns = np.where(sign_change)[0] + 1
    return 2 * (time[turns[-1]] - time[turns[0]]) / (len(turns)-1)

length,theta_0 = 1,np.pi/45
time,theta = simulate_pendulum_better(length,theta_0)
half_period = 0.5 * period_pendulum(time,theta)
print('Half period is','{:.3f}'.format(half_period),'seconds given maximum amplitude of',round(180/np.pi*theta_0),'degrees')

Let's compare periods for different starting angles. 
The larger the starting angle, the longer the period, but the impact is relatively minor for small angles.
Pendulum clocks normally have an amplitude of less than 5 degrees.

In [ ]:
def compare_period_pendulum(length,angles_init,gravity=9.807):
    period = np.zeros(len(angles_init))
    for i in range(len(angles_init)):
        time,theta = simulate_pendulum_better(length,angles_init[i],gravity)
        period[i] =  period_pendulum(time,theta)
    angles_in_degrees = angles_init * 180 / np.pi
    return angles_in_degrees,period
    
length = 1
angles_init = np.linspace(0.001,0.5,50) * np.pi
angle_degrees,period = compare_period_pendulum(length,angles_init)
plt.title('Period of Pendulum')
plt.plot(angle_degrees, period)
plt.xlabel('Starting Angle in degrees')
plt.ylabel('Period in seconds')
plt.figure()
plt.title('Period for starting angle < 20 degrees')
angles_init = np.linspace(0.001,20/180,50) * np.pi
angle_degrees2,period2 = compare_period_pendulum(length,angles_init)
plt.plot(angle_degrees2, period2)
plt.xlabel('Starting Angle in degrees')
plt.ylabel('Period in seconds')
plt.show()  # patience

Measure the daily slowdown for different starting angles.

In [ ]:
def slowdown_per_day(period):
    rel_slowdown = 1 - period[0]/period
    seconds_per_day = 24 * 60 * 60
    slowdown_in_seconds = rel_slowdown * seconds_per_day
    return slowdown_in_seconds

length,num = 1,25
angles_init = np.linspace(0.01,0.25,num) * np.pi
angle_degrees,period = compare_period_pendulum(length,angles_init)
slowdown = slowdown_per_day(period)
print('Slowdown per day relative to starting angle')
for i in range(num):
    a,s = angle_degrees[i],slowdown[i]
    minutes,seconds = divmod(slowdown[i],60) 
    print('{:4.1f}'.format(a),'degrees :',int(minutes),'minutes and',round(seconds),'seconds')